# Análisis de similitud de imágenes con CLIP y FAISS

## Objetivo
Este notebook implementa un flujo de análisis de imágenes basado en embeddings
generados con el modelo CLIP y búsqueda de similitud mediante FAISS.
El objetivo es permitir la recuperación de imágenes visualmente similares
a partir de una imagen o una consulta textual, facilitando la exploración
semántica de archivos visuales y colecciones artísticas.

## Contexto del dataset
Para este ejemplo se trabajó con un corpus de aproximadamente **10.700 imágenes**
provenientes de colecciones abiertas de instituciones culturales, entre ellas:

- **The Art Institute of Chicago**
- **Museo del Banco de la República (Colombia)**
- **Museo Thyssen-Bornemisza**

Las imágenes y sus metadatos fueron unificados en un único dataset estructurado,
permitiendo su análisis conjunto independientemente de la institución de origen.

## Diccionario de datos
El dataset utilizado se organiza como un DataFrame donde cada fila representa
una obra y sus metadatos asociados. Las principales columnas son:

- **id**: Identificador único de la obra (clave de enlace con FAISS).
- **source**: Institución o fuente de procedencia de la obra.
- **source_url**: URL original del registro en la institución fuente.
- **source_url_sha1**: Hash SHA-1 de la URL fuente, usado para deduplicación.
- **inventory_no**: Número de inventario institucional.
- **collection**: Colección curatorial a la que pertenece la obra.
- **room**: Sala o sección de exhibición (cuando aplica).
- **location**: Ubicación física o curatorial de la obra.
- **title**: Título original de la obra.
- **artist**: Autor o artista asociado.
- **date_text**: Fecha de la obra en formato textual.
- **year_start / year_end**: Rango temporal estimado de producción.
- **technique**: Técnica artística empleada.
- **medium**: Materiales o soporte de la obra.
- **dimensions**: Dimensiones físicas.
- **size_text**: Descripción textual del tamaño.
- **description**: Descripción curatorial o interpretativa.
- **info_extra**: Información adicional relevante.
- **bibliography**: Referencias bibliográficas asociadas.
- **exhibition_hist**: Historial de exposiciones.
- **image_url**: URL de la imagen estándar.
- **image_full_url**: URL de la imagen en alta resolución.
- **raw_json**: Registro original en formato JSON.
- **scraped_at**: Fecha de recolección del dato.
- **updated_at**: Fecha de última actualización.
- **title_norm**: Título normalizado para búsquedas.
- **artist_norm**: Nombre del artista normalizado.

## Tecnologías utilizadas
- Python  
- PyTorch  
- CLIP (OpenAI)  
- FAISS  
- Pandas  
- PIL  


## Instalación de dependencias

En esta sección se instalan las librerías necesarias para:
- Manipulación de datos (`pandas`, `numpy`)
- Procesamiento de imágenes (`PIL`)
- Deep Learning (`torch`, `torchvision`)
- Búsqueda vectorial eficiente (`faiss`)


In [5]:
!pip -q install pandas


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
!pip install pandas numpy faiss-cpu


  Obtaining dependency information for faiss-cpu from https://files.pythonhosted.org/packages/87/ff/35ed875423200c17bdd594ce921abfc1812ddd21e09355290b9a94e170ab/faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl.metadata
  Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl (18.9 MB)



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
!pip install torch torchvision torchaudio


  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/6e/01/624c4324ca01f66ae4c7cd1b74eb16fb52596dce66dbe51eff95ef9e7a4c/torch-2.10.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/ad/16/8f650c2e288977cf0f8f85184b90ee56ed170a4919347fc74ee99286ed6f/torchvision-0.25.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://files.pythonhosted.org/packages/be/a0/da53c7d20fac15f66f8838653b91162de1bf21fb40fee88cf839e4ef5174/torchaudio-2.10.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for filelock from https://files.pythonhosted.org/packages/b5/36/7fb70f04bf00bc646cd5bb45aa9eddb15e19437a28b8fb2b4a5249fac770/filelock-3.20.3-py3-none-any.whl.metadata
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packag


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip -q install pillow


In [36]:
%%capture
import pandas as pd
df = pd.read_csv(
    "obras_arte.csv",
    encoding="utf-8"
)

print(df.head())
print(df.shape)

In [37]:
df

,id,source,source_url,source_url_sha1,inventory_no,collection,room,location,title,artist,...,info_extra,bibliography,exhibition_hist,image_url,image_full_url,raw_json,scraped_at,updated_at,title_norm,artist_norm
0,4428,thyssen,https://www.museothyssen.org/coleccion/artista...,b'\xd2\x9f\xcc\x04\xfb\n\x1eQ6\xe8(4m\xeb\x06\...,Nº INV. ( DEC1656 ),NaN,Sala 2,"Thyssen-Bornemisza Collections, en depósito en...",La Virgen y el Niño,Anónimo alemán,...,Esta pequeña escultura devocional en madera co...,NaN,NaN,https://www.museothyssen.org/sites/default/fil...,NaN,NaN,2026-01-20 23:47:41,2026-01-20 23:47:41,la virgen y el niño,anónimo alemán
1,4429,thyssen,https://www.museothyssen.org/coleccion/artista...,b'\x0c\x8e\xcbC\x02\x86-9|\xb7P\xbd\xab?2\xe0\...,Nº INV. 44.a ( 1931.4 ),NaN,Sala 2,"Museo Nacional Thyssen-Bornemisza, Madrid",Tríptico de la Santa Faz,Maestro Bertram,...,"Pintor, escultor e iluminador alemán del cambi...",NaN,NaN,https://www.museothyssen.org/sites/default/fil...,NaN,NaN,2026-01-20 23:47:41,2026-01-20 23:47:41,tríptico de la santa faz,maestro bertram
2,4430,thyssen,https://www.museothyssen.org/coleccion/artista...,b'>\x83\x17O\x05\xd4\x8f\r\xad\x95J[\x96\x8fo\...,Nº INV. 151 ( 1977.98 ),NaN,Sala 1,"Museo Nacional Thyssen-Bornemisza, Madrid",La Crucifixión,Agnolo Gaddi,...,Agnolo Gaddi es uno de los mejores representan...,NaN,NaN,https://www.museothyssen.org/sites/default/fil...,NaN,NaN,2026-01-20 23:47:41,2026-01-20 23:47:41,la crucifixión,agnolo gaddi
3,4431,thyssen,https://www.museothyssen.org/coleccion/artista...,b'\xc9\x9e5u\xe6d\x92\x97k\xfd>f\x92J1\xefV\x1...,Nº INV. 44.d-e ( 1929.19.1-2.b ),NaN,Sala 2,"Museo Nacional Thyssen-Bornemisza, Madrid",Tríptico de la Santa Faz (cerrado): La Anuncia...,Maestro Bertram,...,"Pintor, escultor e iluminador alemán del cambi...",NaN,NaN,https://www.museothyssen.org/sites/default/fil...,NaN,NaN,2026-01-20 23:47:41,2026-01-20 23:47:41,tríptico de la santa faz (cerrado): la anuncia...,maestro bertram
4,4432,thyssen,https://www.museothyssen.org/coleccion/artista...,b'\xa1\xec\x05\xd5\xc93\xf9\xc8I\xfd\x1b\xe3\x...,Nº INV. 11 ( 1956.11 ),NaN,Sala 5,"Museo Nacional Thyssen-Bornemisza, Madrid","Retrato póstumo de Wenceslao de Luxemburgo, du...",Anónimo franco flamenco,...,El personaje representado se ha podido identif...,NaN,NaN,https://www.museothyssen.org/sites/default/fil...,NaN,NaN,2026-01-20 23:47:41,2026-01-20 23:47:41,"retrato póstumo de wenceslao de luxemburgo, du...",anónimo franco flamenco
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10726,15732,ARTIC,https://www.artic.edu/artworks/28697/woman-fig...,b'(\xc8oA\xe2\x16\xd0\xa8\xff\xf8)\xab&0\xdd\x...,NaN,NaN,NaN,Chicago,"Woman Figure with Medusa Ornaments, 1967",Claes Oldenburg,...,NaN,NaN,NaN,https://www.artic.edu/iiif/2/4d5080ec-3718-49e...,https://www.artic.edu/iiif/2/4d5080ec-3718-49e...,NaN,2026-01-21 18:42:39,2026-01-21 18:42:39,"woman figure with medusa ornaments, 1967",claes oldenburg
10727,15733,ARTIC,https://www.artic.edu/artworks/32060/i-coming-...,b'\xf5\\7\xff\xe8GN\xe5~\xa9i\x9b\x9aW\xcb\xd6...,NaN,NaN,NaN,Chicago,"“I”: Coming of Age, from the series “Tales of ...",Katsukawa Shunsho,...,NaN,NaN,NaN,https://www.artic.edu/iiif/2/fa5b7e62-06ed-1de...,https://www.artic.edu/iiif/2/fa5b7e62-06ed-1de...,NaN,2026-01-21 18:42:40,2026-01-21 18:42:40,"“i”: coming of age, from the series “tales of ...",katsukawa shunsho
10728,15734,ARTIC,https://www.artic.edu/artworks/59924/albatross...,b'pE9m\\G/\x9c\xbd\xf1\x08\xa8\x1b\x88A\x0cO\x...,NaN,NaN,NaN,Chicago,"Albatrossi (Albatross) (Furnishing Fabric), 1967",Maija Isola,...,"Art Institute of Chicago, European Decorative ...",NaN,NaN,https://www.artic.edu/iiif/2/f1a34741-98cb-7fd...,https://www.artic.edu/iiif/2/f1a34741-98cb-7fd...,NaN,2026-01-21 18:42:40,2026-01-21 18:42:40,"albatrossi (albatross) (furnishing fabric), 1967",maija isola
10729,15735,ARTIC,https://www.artic.edu/artworks/129118/so-then-...,"b'\xe5\xabr\xee6\xb1S\xa2\r

In [6]:
df.columns

Index(['id', 'source', 'source_url', 'source_url_sha1', 'inventory_no',
       'collection', 'room', 'location', 'title', 'artist', 'date_text',
       'year_start', 'year_end', 'technique', 'medium', 'dimensions',
       'size_text', 'description', 'info_extra', 'bibliography',
       'exhibition_hist', 'image_url', 'image_full_url', 'raw_json',
       'scraped_at', 'updated_at', 'title_norm', 'artist_norm'],
      dtype='str')

## Carga del índice FAISS persistido

En esta sección se carga desde disco un índice FAISS previamente generado.
Este índice contiene los embeddings de las imágenes calculados con el modelo CLIP
y permite realizar búsquedas de similitud sin necesidad de recalcularlos.

El uso de un índice persistido:
- Reduce significativamente el tiempo de ejecución
- Permite reutilizar el modelo en distintos notebooks o aplicaciones
- Facilita la integración con sistemas externos (APIs, Streamlit, Web)

El archivo `.index` fue generado previamente y se carga utilizando `faiss.read_index`.

El índice FAISS almacena únicamente los vectores numéricos y sus posiciones,
por lo que la relación con las imágenes originales se mantiene mediante
una estructura externa de metadatos (por ejemplo, una lista o base de datos).


In [7]:
import os
import faiss

INDEX_PATH = r"C:\Users\David\PycharmProjects\thyssen\clip_vitb32.index"  # <-- cambia esto

INDEX_PATH = os.path.abspath(INDEX_PATH)
index = faiss.read_index(INDEX_PATH)
print("✅ Index cargado:", INDEX_PATH)
print("ntotal:", index.ntotal)


✅ Index cargado: C:\Users\David\PycharmProjects\thyssen\clip_vitb32.index
ntotal: 9957


## Recuperación de identificadores almacenados en FAISS

Los índices FAISS de tipo `IndexIDMap` o `IndexIDMap2` permiten asociar cada
vector almacenado con un identificador externo (por ejemplo, un ID de base
de datos, un índice de archivo o un identificador lógico).

En esta sección se recuperan los IDs persistidos dentro del índice FAISS.
Esto es fundamental para:
- Reconectar los resultados de la búsqueda con los archivos de imagen originales
- Mantener la trazabilidad entre vectores y metadatos externos
- Integrar FAISS con bases de datos o sistemas de archivo

Si el índice no fue creado como `IDMap`, esta operación no es posible y se
lanza una excepción.


Nota: En FAISS, el orden de los IDs recuperados corresponde al orden interno
de los vectores en el índice. Por esta razón, es indispensable mantener una
estructura externa de metadatos alineada con estos identificadores.



In [8]:
import numpy as np
import faiss

def get_faiss_ids(index) -> np.ndarray:
    # Para IndexIDMap / IndexIDMap2, FAISS guarda los ids en index.id_map
    if hasattr(index, "id_map"):
        return faiss.vector_to_array(index.id_map).astype(np.int64)
    raise TypeError("Este índice no tiene id_map (no es IDMap).")

ids_faiss = get_faiss_ids(index)
print("IDs en FAISS:", ids_faiss[:10], " ... total:", len(ids_faiss))


IDs en FAISS: [4455 4456 4457 4462 4463 4464 4465 4466 4467 4468]  ... total: 9957


## Alineación del índice FAISS con los metadatos del dataset

FAISS almacena únicamente vectores y sus identificadores, pero no conserva
información contextual como rutas de archivo, nombres o fuentes.
Por esta razón, es necesario reconstruir la relación entre los embeddings
y los metadatos originales.

En esta sección se:
1. Asegura que la columna `id` del DataFrame tenga un tipo numérico consistente
2. Reordena el DataFrame según el orden interno de los IDs almacenados en FAISS
3. Garantiza que cada vector del índice tenga su fila correspondiente en el dataset

El uso de `reindex(ids_faiss)` es fundamental para mantener la coherencia
entre los resultados de búsqueda y la información asociada a cada imagen.

Nota: La presencia de valores `NaN` indica que existen embeddings en el índice
FAISS que no tienen correspondencia directa en el archivo de metadatos.
Esto puede ocurrir si el índice fue generado a partir de múltiples fuentes
o si el dataset fue filtrado posteriormente.



In [9]:
import pandas as pd
import numpy as np

# asegura tipo
df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")

# ordena según ids_faiss
df_aligned = (
    df.set_index("id")
      .reindex(ids_faiss)          # respeta el orden del FAISS
      .reset_index()
)

# filas que no existan en df quedan como NaN (por si el CSV no tiene todas)
print(df_aligned.head())
print("Filas alineadas:", len(df_aligned))
print("Faltantes en df:", df_aligned["source"].isna().sum())


     id   source                                         source_url  \
0  4455  thyssen  https://www.museothyssen.org/coleccion/artista...   
1  4456  thyssen  https://www.museothyssen.org/coleccion/artista...   
2  4457  thyssen  https://www.museothyssen.org/coleccion/artista...   
3  4462  thyssen  https://www.museothyssen.org/coleccion/artista...   
4  4463  thyssen  https://www.museothyssen.org/coleccion/artista...   

                                     source_url_sha1  \
0  b'S\xb1~t\xfb\xb8\x07\n\xaaQ\xa4S\xd1\x11\xdf~...   
1  b'_\x1dW\xac\xdba\x03\x89\x94S)jl\xa6\x96=!\x8...   
2  b'\xedh\xf6&\xb6\x9f\xc0\xeeD\x8dP2\xa4\xf9,\x...   
3  b' \xe5\xd3\xe9u;\x14{\xeb\x94qzO\x06\xde\x0f\...   
4  b'\xed\xe6~\xf2\xe6\xdbJ#T:\xc6<\x90\xcaa3\x90...   

               inventory_no collection                room  \
0   Nº INV. 210 ( 1929.13 )        NaN              Sala 2   
1  Nº INV. 411 ( 1930.118 )        NaN              Sala 4   
2    Nº INV. 162 ( 1966.6 )        NaN            

## Filtrado del dataset a elementos presentes en el índice FAISS

En esta sección se construye un subconjunto del DataFrame original que
contiene únicamente las obras que están representadas dentro del índice FAISS.

Este filtrado permite:
- Evitar inconsistencias entre el dataset y el índice vectorial
- Asegurar que todas las filas del DataFrame tengan un embedding asociado
- Simplificar procesos posteriores de búsqueda y visualización

El criterio de inclusión se basa en la pertenencia del `id` al conjunto de
identificadores almacenados en FAISS.


In [10]:
df_in = df[df["id"].isin(ids_faiss)].copy()
print("Solo obras presentes en FAISS:", df_in.shape)


Solo obras presentes en FAISS: (9957, 28)


## Marcado de registros presentes en el índice FAISS

En esta sección se crea una columna indicadora (`in_faiss`) que señala si cada
registro del DataFrame tiene un embedding almacenado en el índice FAISS.

Este enfoque permite:
- Mantener el dataset completo sin descartar información
- Diferenciar claramente qué obras participan en la búsqueda vectorial
- Facilitar análisis comparativos, filtros dinámicos o visualizaciones

La columna `in_faiss` toma el valor:
- `1` si el `id` del registro está presente en FAISS
- `0` en caso contrario


In [11]:
s = set(ids_faiss.tolist())
df["in_faiss"] = df["id"].astype("Int64").apply(lambda x: int(x in s) if pd.notna(x) else 0)


## Búsqueda texto → imagen con CLIP + FAISS

En esta sección se habilita la búsqueda de obras a partir de una consulta en texto
(por ejemplo: “paisaje nocturno”, “retrato cubista”, “caballo”, etc.).

El flujo es:

1. **Carga del modelo CLIP** (`openai/clip-vit-base-patch32`) y selección automática de dispositivo
   (`cuda` si hay GPU disponible, si no `cpu`).
2. **Generación del embedding del texto**:
   - Se tokeniza el texto con `CLIPProcessor`
   - Se obtiene la representación del encoder de texto (`model.text_model`)
   - Se proyecta al espacio común de CLIP (512 dimensiones) con `text_projection`
   - Se **normaliza** el vector para que la similitud por producto punto (IP) sea equivalente a coseno
3. **Consulta en FAISS**:
   - `index.search(q, k)` devuelve los `k` resultados más similares
   - `I` contiene los IDs (o posiciones) recuperados
   - `D` contiene los scores de similitud
4. **Reconstrucción con metadatos**:
   - Se mapean los IDs recuperados al DataFrame (`df`) para devolver resultados interpretables
   - Se filtran IDs que no existan en el DataFrame, si los hubiera


In [32]:
import numpy as np
import pandas as pd
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

def text_embedding(text: str) -> np.ndarray:
    tok = processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    tok = {k: v.to(device) for k, v in tok.items()}

    with torch.inference_mode():
        # 1) pasa SOLO por el encoder de texto
        text_out = model.text_model(
            input_ids=tok["input_ids"],
            attention_mask=tok.get("attention_mask", None),
        )
        # 2) pooler_output -> proyección CLIP a 512
        v = text_out.pooler_output
        v = model.text_projection(v)

        # 3) normalizar para IndexFlatIP
        v = v / v.norm(dim=-1, keepdim=True)

    return v.detach().cpu().numpy().astype("float32")  # (1,512)

def search_df(query: str, k: int = 10) -> pd.DataFrame:
    q = text_embedding(query)             # (1,512)
    D, I = index.search(q, k)             # I: (1,k) ids de obras, D: scores
    ids = I[0].astype("int64")
    scores = D[0]

    # OJO: si algún id no está en df, lo filtramos
    df_idx = df.set_index("id")
    keep = [int(x) for x in ids if int(x) in df_idx.index]

    out = (df_idx.loc[keep]
                 .assign(score=scores[:len(keep)])
                 .reset_index()
                 .sort_values("score", ascending=False))
    return out


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     | Details
-------------------------------------+------------+--------
vision_model.embeddings.position_ids | UNEXPECTED |        
text_model.embeddings.position_ids   | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Ejecución de la consulta y visualización de resultados

En esta sección se permite al usuario introducir una consulta textual que será
utilizada para buscar obras visualmente relacionadas dentro del índice FAISS.

El flujo es el siguiente:
- El usuario ingresa un texto libre (consulta semántica)
- Se ejecuta la función de búsqueda `search_df`
- Se recuperan las `k` obras más similares
- Se muestran los principales campos descriptivos junto con el score de similitud

El valor `score` representa el grado de similitud entre el embedding del texto
y el embedding de la imagen; valores más altos indican mayor afinidad semántica.

Este mecanismo demuestra cómo es posible consultar un archivo visual a partir
de lenguaje natural, habilitando nuevas formas de exploración, curaduría y
análisis semántico en colecciones de imágenes.



In [40]:
qtext = input("🔎 Buscar: ")   
df_results = search_df(qtext, k=15)

df_results[["id","title","artist","score","image_url"]].head(15)

🔎 Buscar:  dancing


,id,title,artist,score,image_url
0,9565,"Dancer Stretching at the Bar, 1877/80",Hilaire Germain Edgar Degas,0.284347,https://www.artic.edu/iiif/2/1178d05a-3713-a50...
1,8196,"The Dream of Saint Jerome, 1476",Matteo di Giovanni,0.283338,https://www.artic.edu/iiif/2/336a83f3-c184-add...
2,8507,"Two Dancers, c. 1893–98",Hilaire Germain Edgar Degas,0.282555,https://www.artic.edu/iiif/2/1381bcfe-b5a9-777...
3,11387,"Four Women Dancing, c. 1497",Andrea Mantegna,0.281230,https://www.artic.edu/iiif/2/8213a708-ef00-761...
4,6648,"Ballet at the Paris Opéra, 1877",Hilaire Germain Edgar Degas,0.272941,https://www.artic.edu/iiif/2/cb34b0a8-bc51-d06...
5,10436,"Dancer Bending Forward, 1874/79",Hilaire Germain Edgar Degas,0.272922,https://www.artic.edu/iiif/2/fdb26db9-c4cd-96f...
6,7378,"The Star, 1879/81",Hilaire Germain Edgar Degas,0.271021,https://www.artic.edu/iiif/2/0850f3ab-a29d-acc...
7,8581,"Be Careful with that Step!, 1816/1820",Francisco José de Goya y Lucientes,0.271017,https://www.artic.edu/iiif/2/626bcfe3-6a1b-281...
8,6701,"Croquet Scene, 1866",Winslow Homer,0.269593,https://www.artic.edu/iiif/2/f7b2db45-f393-653...
9,7174,"Alice, 1892",William Merritt Chase,0.268446,https://www.artic.edu/iiif/2/91fcc71a-50eb-f41...


## Visualización de los resultados recuperados

En esta sección se presentan visualmente las obras más relevantes devueltas por
la búsqueda semántica texto → imagen.

Para cada resultado se muestra:
- El título de la obra
- El artista
- El score de similitud semántica
- La imagen asociada (si está disponible)

Esta visualización permite evaluar de forma cualitativa la coherencia entre
la consulta textual y los resultados obtenidos, complementando el análisis
numérico con una inspección visual directa.


In [41]:
from IPython.display import Image, display

top = df_results.head(10)

for i, row in top.iterrows():
    print(f"{row['title']} — {row['artist']} (score={row['score']:.3f})")
    if pd.notna(row["image_url"]):
        display(Image(url=row["image_url"], width=300))
    print("-" * 60)


Dancer Stretching at the Bar, 1877/80 — Hilaire Germain Edgar Degas (score=0.284)


------------------------------------------------------------
The Dream of Saint Jerome, 1476 — Matteo di Giovanni (score=0.283)


------------------------------------------------------------
Two Dancers, c. 1893–98 — Hilaire Germain Edgar Degas (score=0.283)


------------------------------------------------------------
Four Women Dancing, c. 1497 — Andrea Mantegna (score=0.281)


------------------------------------------------------------
Ballet at the Paris Opéra, 1877 — Hilaire Germain Edgar Degas (score=0.273)


...

------------------------------------------------------------
The Star, 1879/81 — Hilaire Germain Edgar Degas (score=0.271)


------------------------------------------------------------
Be Careful with that Step!, 1816/1820 — Francisco José de Goya y Lucientes (score=0.271)


------------------------------------------------------------
Croquet Scene, 1866 — Winslow Homer (score=0.270)


------------------------------------------------------------
Alice, 1892 — William Merritt Chase (score=0.268)


------------------------------------------------------------


## Búsqueda imagen → imagen con CLIP + FAISS

En esta sección se implementa la recuperación de imágenes similares a partir de
una imagen de consulta (búsqueda visual).

### Idea principal
1. Se toma una **imagen local** como entrada.
2. Se genera su **embedding CLIP** (vector de 512 dimensiones) usando el encoder visual.
3. Se **normaliza** el embedding para que el producto punto (Inner Product) se comporte
   como similitud coseno (esto es lo habitual cuando el índice FAISS es `IndexFlatIP`).
4. Se consulta el índice FAISS (`index.search`) para recuperar las `k` imágenes más similares.
5. Se reconstruyen los resultados usando los **metadatos** almacenados en el DataFrame (`df`).

### Funciones incluidas
- `image_embedding_from_path(img_path)`: genera el embedding de una imagen local y lo normaliza.
- `search_by_image(img_path, k)`: realiza la búsqueda en FAISS y retorna un DataFrame con:
  `id`, `title`, `artist`, `image_url` y `score`.

### Nota sobre IDs y metadatos
FAISS no almacena metadatos (título, artista, URL).  
Por eso, los resultados deben mapearse a `df` mediante el campo `id`.
El mapeo asume un caso típico donde el índice es un `IDMap` y devuelve IDs reales
que coinciden con `df["id"]`. Si el índice devuelve posiciones internas, se requiere
una tabla de correspondencia adicional.


In [44]:
# =========================
# IMAGEN -> IMAGEN (CLIP + FAISS)
# =========================

import numpy as np
import pandas as pd
import torch
from PIL import Image

# Requisitos previos (ya los tienes en tu notebook):
# - df: DataFrame con columna 'id' (int) y campos como title/artist/image_url
# - index: índice FAISS ya cargado
# - model, processor: CLIPModel/CLIPProcessor cargados
# - device: "cuda" o "cpu"

def image_embedding_from_path(img_path: str) -> np.ndarray:
    """Genera embedding CLIP (1,512) para una imagen local y lo normaliza."""
    img = Image.open(img_path).convert("RGB")

    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # capa de proyección (según versión/modelo)
    proj = getattr(model, "visual_projection", None) or getattr(model, "vision_projection", None)
    if proj is None:
        raise AttributeError("No encuentro visual_projection/vision_projection en el modelo CLIP.")

    with torch.inference_mode():
        # encoder de visión (salida completa)
        vision_out = model.vision_model(**inputs)

        # pooler_output -> proyección a 512
        v = proj(vision_out.pooler_output)  # (1,512)

        # normalizar para IndexFlatIP (cosine)
        v = v / v.norm(dim=-1, keepdim=True)

    return v.detach().cpu().numpy().astype("float32")  # (1,512)

def search_by_image(img_path: str, k: int = 10) -> pd.DataFrame:
    """
    Busca las k más similares en el índice FAISS usando una imagen externa.
    Devuelve un DataFrame con score y metadatos del df.
    """
    q = image_embedding_from_path(img_path)   # (1,512)
    D, I = index.search(q, k)                 # I: (1,k) ids o posiciones; D: (1,k) scores

    ids = I[0].astype("int64")
    scores = D[0]

    # Caso típico: tu índice es IDMap y devuelve ids reales = df['id']
    df_idx = df.set_index("id")
    keep_ids = [int(x) for x in ids if int(x) in df_idx.index]

    out = (df_idx.loc[keep_ids]
           .assign(score=scores[:len(keep_ids)])
           .reset_index()
           .sort_values("score", ascending=False))
    return out

# ---- USO ----
img_path = r"C:\Users\David\PycharmProjects\proyectos_organizados\humanidades_digitales\analisis_de_imagen\c93000213c2270e296338c28c4f668be.jpg"
df_sim = search_by_image(img_path, k=15)
display(df_sim[["id","title","artist","score","image_url"]].head(15))

# ---- MOSTRAR TOP IMÁGENES (opcional) ----
from IPython.display import Image as IPyImage, display

top = df_sim.head(10)
for _, row in top.iterrows():
    print(f"{row.get('title','')} — {row.get('artist','')} (score={row['score']:.3f})")
    if pd.notna(row.get("image_url", None)):
        display(IPyImage(url=row["image_url"], width=300))
    print("-" * 60)





,id,title,artist,score,image_url
0,12720,"Plate, 1780–1820",Artist unknown,0.758020,https://www.artic.edu/iiif/2/30094675-1d8b-c59...
1,11232,"Star-Shaped Tile with Qur’anic Inscriptions, I...",Islamic,0.752142,https://www.artic.edu/iiif/2/49462042-626d-f56...
2,11988,Prutah (Coin) Depicting an Inscription within ...,Judean,0.749460,https://www.artic.edu/iiif/2/976146af-04f9-110...
3,11976,Schiller Building (later Garrick Theater): Sec...,Louis H. Sullivan,0.743294,https://www.artic.edu/iiif/2/e03b2df9-381d-ddc...
4,12020,"The Reader, 1919",Jacques Lipchitz,0.741783,https://www.artic.edu/iiif/2/975ad8c7-1035-70c...
5,10517,"Black Panel, 1989/99",Ellsworth Kelly,0.738873,https://www.artic.edu/iiif/2/e763fafc-fc52-6f6...
6,10181,"Landscape 4, from Ten Landscapes, 1967",Roy Lichtenstein,0.736983,https://www.artic.edu/iiif/2/a610c6c1-ed95-2a1...
7,9172,Elevator T-Plate from the Chicago Stock Exchan...,"Adler & Sullivan, Architects",0.733938,https://www.artic.edu/iiif/2/4dc63cce-a3c0-997...
8,14899,"Black and White LOVE, from Decade, 1971",Robert Indiana,0.727979,https://www.artic.edu/iiif/2/d646cff4-a21f-bf6...
9,8867,"The First Knot, c. 1507",Albrecht Dürer,0.727614,https://www.artic.edu/iiif/2/eed05c65-f9ea-ecc...


Plate, 1780–1820 — Artist unknown (score=0.758)


------------------------------------------------------------
Star-Shaped Tile with Qur’anic Inscriptions, Ilkhanid dynasty (1256–1353), Mid-13th century — Islamic (score=0.752)


------------------------------------------------------------
Prutah (Coin) Depicting an Inscription within wreath, Hasmonean Dynasty (between 135 BCE-35 BCE) — Judean (score=0.749)


------------------------------------------------------------
Schiller Building (later Garrick Theater): Sections of Star-Pod Design from Proscenium Vault, c. 1891/92 — Louis H. Sullivan (score=0.743)


...

------------------------------------------------------------
Landscape 4, from Ten Landscapes, 1967 — Roy Lichtenstein (score=0.737)


------------------------------------------------------------
Elevator T-Plate from the Chicago Stock Exchange, 1894 — Adler & Sullivan, Architects (score=0.734)


------------------------------------------------------------
Black and White LOVE, from Decade, 1971 — Robert Indiana (score=0.728)


------------------------------------------------------------
The First Knot, c. 1507 — Albrecht Dürer (score=0.728)


------------------------------------------------------------
